In [ ]:
# print("123")

123


In [1]:
!pip install opentelemetry-api

In [2]:
!pip install opentelemetry-sdk

In [3]:
!pip install python-dotenv

In [4]:
!pip install gitsource

In [5]:
!pip install minsearch

In [6]:
!pip install google-genai

In [9]:
import os
import sqlite3
import time
import pandas as pd

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
    SpanExporter,
    SpanExportResult,
)
from starter import rag

In [2]:
# instrumentation created at import time is backed by our provider.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
INPUT_PRICE_PER_MILLION = 0.15
OUTPUT_PRICE_PER_MILLION = 0.60

In [ ]:
class RAGTraced:
    """Wraps a RAGBase instance so rag(), search(), and llm() each
    produce their own OTel span."""

    def __init__(self, rag_base, tracer):
        self._rag = rag_base
        self._tracer = tracer

    def search(self, query, num_results=5):
        with self._tracer.start_as_current_span("search") as span:
            results = self._rag.search(query, num_results=num_results)
            span.set_attribute("num_results", len(results))
            return results

    def llm(self, prompt):
        with self._tracer.start_as_current_span("llm") as span:
            response = self._rag.llm(prompt)
            
            usage = response.usage_metadata
            input_tokens = usage.prompt_token_count
            output_tokens = usage.candidates_token_count

            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)

            input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
            output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
            cost = input_cost + output_cost
            span.set_attribute("cost", cost)

            return response

    def build_prompt(self, query, search_results):
        return self._rag.build_prompt(query, search_results)

    def rag(self, query):
        with self._tracer.start_as_current_span("rag") as span:
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            
            return response.text

In [5]:
QUERY = "How does the agentic loop keep calling the model until it stops?"

In [6]:
traced_rag = RAGTraced(rag, tracer)

In [7]:
print("=" * 70)
print("Q1-Q3: Running traced RAG with ConsoleSpanExporter")
print("=" * 70)

answer = traced_rag.rag(QUERY)
provider.force_flush()

print("\nAnswer:", answer[:200], "...")
print(
    "\n(Inspect the ReadableSpan dicts printed above to answer Q1 [span count], "
    "Q2 [llm span's input_tokens attribute], and Q3 [search vs llm span duration])"
)

Q1-Q3: Running traced RAG with ConsoleSpanExporter
{
    "name": "search",
    "context": {
        "trace_id": "0x869263d813e17199bf61e2d0c583577a",
        "span_id": "0x6995940644c5f097",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2c0c360029ca056f",
    "start_time": "2026-07-27T08:15:28.937375Z",
    "end_time": "2026-07-27T08:15:28.939430Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "64cc6601-c92d-4260-9fab-8d663ce3fdaa",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x869263d813e17199bf61e2d0c583577a",
        "span_id

In [10]:
class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [11]:
db_path = "traces.db"
if os.path.exists(db_path):
    os.remove(db_path)

sqlite_provider = TracerProvider()
sqlite_provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter(db_path)))
sqlite_tracer = sqlite_provider.get_tracer("llm-zoomcamp")
sqlite_traced_rag = RAGTraced(rag, sqlite_tracer)

print("\n" + "=" * 70)
print("Q4: Re-running the query with the SQLite exporter")
print("=" * 70)

sqlite_traced_rag.rag(QUERY)
sqlite_provider.force_flush()

conn = sqlite3.connect(db_path)
span_names = pd.read_sql("SELECT DISTINCT name FROM spans ORDER BY name", conn)["name"].tolist()
conn.close()
print(f"Span names in the spans table: {span_names}")


Q4: Re-running the query with the SQLite exporter
Span names in the spans table: ['llm', 'rag', 'search']


In [13]:
with sqlite3.connect("traces.db") as conn:
    for row in conn.execute("SELECT * FROM spans"):
        print(row)

('search', 1785142724994277000, 1785142724998055900, None, None, None)
('llm', 1785142725004873800, 1785142729157248700, 7987, 273, 0.0013618499999999997)
('rag', 1785142724994238300, 1785142729163262200, None, None, None)


In [14]:
query = """
SELECT name, SUM(end_time - start_time) as total_duration
FROM spans
WHERE name != 'rag'
GROUP BY name
ORDER BY total_duration DESC
"""
print("total_duration")
with sqlite3.connect("traces.db") as conn:
    for row in conn.execute(query):
        print(f"{row[0]}: {row[1]/1_000_000:.1f} ms")

total_duration
llm: 4152.4 ms
search: 3.8 ms


In [12]:
print("\n" + "=" * 70)
print("Q5: Total duration by span name (excluding 'rag')")
print("=" * 70)

conn = sqlite3.connect(db_path)
df_spans = pd.read_sql("SELECT * FROM spans", conn)
conn.close()

df_spans["duration_ms"] = (df_spans["end_time"] - df_spans["start_time"]) / 1_000_000

duration_by_name = (
    df_spans[df_spans["name"] != "rag"]
    .groupby("name")["duration_ms"]
    .sum()
    .sort_values(ascending=False)
)
print(duration_by_name)
q5_top_span = duration_by_name.idxmax()
print(f"\nSpan type with most total time: {q5_top_span}")


Q5: Total duration by span name (excluding 'rag')
name
llm       4152.3749
search       3.7789
Name: duration_ms, dtype: float64

Span type with most total time: llm


In [15]:
query = """
SELECT input_tokens
FROM spans
WHERE name = 'rag'
"""

with sqlite3.connect("traces.db") as conn:
    df = pd.read_sql_query(query, conn)

# How much do the input tokens vary across these 4 runs
variation = (df.input_tokens.max()-df.input_tokens.min())/df.input_tokens.mean()
print(f"input_tokens variation is {variation:.1%} across these 4 runs")

input_tokens variation is nan% across these 4 runs


In [16]:
print("\n" + "=" * 70)
print("Q6: Running the same query 3 more times for token-stability check")
print("=" * 70)

for _ in range(3):
    sqlite_traced_rag.rag(QUERY)
sqlite_provider.force_flush()

conn = sqlite3.connect(db_path)
df_llm = pd.read_sql(
    "SELECT input_tokens FROM spans WHERE name = 'llm' ORDER BY start_time", conn
)
conn.close()

tokens = df_llm["input_tokens"].tolist()
print(f"Input tokens across {len(tokens)} runs: {tokens}")

min_tok, max_tok = min(tokens), max(tokens)
variance_pct = ((max_tok - min_tok) / min_tok) * 100 if min_tok else 0.0
print(f"Min: {min_tok}, Max: {max_tok}, variance: {variance_pct:.2f}%")

if variance_pct == 0:
    q6_answer = "They're identical"
elif variance_pct <= 10:
    q6_answer = "Within 10% of each other"
elif variance_pct <= 50:
    q6_answer = "Within 50% of each other"
else:
    q6_answer = "They vary more than 50%"


Q6: Running the same query 3 more times for token-stability check
Input tokens across 4 runs: [7987, 7987, 7987, 7987]
Min: 7987, Max: 7987, variance: 0.00%
